In [ ]:
from scipy import io
import numpy as np
dataslices_output='data_slices/output'
batch_size=1

audio_dir='../../assets/trainingdata/chords/'
sampleRate,audio=io.wavfile.read(audio_dir+'session_original.wav')
audio=audio.astype(np.float32)
_,audio_hi=io.wavfile.read(audio_dir+'session_eq high.wav')
audio_hi=audio_hi.astype(np.float32)
_,audio_mid=io.wavfile.read(audio_dir+'session_eq mid.wav')
audio_mid=audio_mid.astype(np.float32)
_,audio_dist=io.wavfile.read(audio_dir+'session_tape dist.wav')
audio_dist=audio_dist.astype(np.float32)

use_augmentations=False


In [ ]:
import numpy as np
import seaborn as sns
from scipy import signal
from fretboard import FretBoard
    
    
filter =FretBoard(20,sampleRate)
numfilters=filter.get_num_filters()
filterbank_out=np.zeros((numfilters,len(audio)),dtype=np.float32)
filter.process(audio,filterbank_out)


if use_augmentations:
    print('Processing Augmentations')
    filter_hi =FretBoard(20,sampleRate)
    filter_mid =FretBoard(20,sampleRate)
    filter_dist =FretBoard(20,sampleRate)
    print('num filters:'+str(numfilters)+' audio samples '+str(len(audio)))
    
    
    
    filterbank_out_hi=np.zeros((numfilters,len(audio)),dtype=np.float32)
    filterbank_out_mid=np.zeros((numfilters,len(audio)),dtype=np.float32)
    filterbank_out_dist=np.zeros((numfilters,len(audio)),dtype=np.float32)
    
    
    
    
    filter_hi.process(audio_hi,filterbank_out_hi)
    filter_mid.process(audio_mid,filterbank_out_mid)
    filter_dist.process(audio_dist,filterbank_out_dist)
else: 
    print('NOT Processing augmentations')

NOT Processing augmentations


In [3]:
from common import frame_size
def reshape_to_nn_input(indata):
    num_cols=indata.shape[1]
    num_rows=indata.shape[0]
    downsample_factor = frame_size
    # --- Downsampling the data ---
    print(f"reshape data by a factor of {downsample_factor}...")
    # Calculate the new number of columns after downsampling
    new_num_cols = num_cols // downsample_factor

    # Ensure the original number of columns is a multiple of the downsample_factor
    # If not, you might lose some data at the end or need a more complex aggregation.
    # For simplicity, we'll slice to a multiple of downsample_factor
    effective_cols = new_num_cols * downsample_factor
    data_sliced = indata[:, :effective_cols]
    print(data_sliced.shape)
    # Reshape the data for averaging:
    # -1: infer dimension
    # downsample_factor: group columns into blocks
    # num_rows: keep rows as isp
    # This reshapes (19, M*N) to (19, M, N)
    reshaped_data = data_sliced.reshape(num_rows, new_num_cols, downsample_factor,1)
    reshaped_data=np.swapaxes(reshaped_data,0,1)
    reshaped_data=np.swapaxes(reshaped_data,1,2)
    
    print('Reshaped the input data to  ')
    print(reshaped_data.shape)
    return reshaped_data
nn_input=reshape_to_nn_input(filterbank_out)
filterbank_out=None
if use_augmentations:
    nn_input_all=np.concatenate((nn_input,nn_input,nn_input,nn_input),axis=0)
    nsamples=nn_input.shape[0];
    nn_input_all[range(nsamples,2*nsamples)]=reshape_to_nn_input(filterbank_out_hi)
    filterbank_out_hi=None
    
    nn_input_all[range(2*nsamples,3*nsamples)]=reshape_to_nn_input(filterbank_out_mid)
    filterbank_out_mid=None
    
    nn_input_all[range(3*nsamples,4*nsamples)]=reshape_to_nn_input(filterbank_out_dist)
    filterbank_out_dist=None
    
    
    print(nn_input_all.shape)

reshape data by a factor of 256...
(312, 12585984)
Reshaped the input data to  
(49164, 256, 312, 1)


In [4]:
from common import save_data_slices
output_dir_input = 'data_slices/input'
if use_augmentations:
    save_data_slices(output_dir_input,nn_input_all,batch_size)
else:
    save_data_slices(output_dir_input,nn_input,batch_size)

Saving 49164 samples to disk
Serialization complete. 49164 Files saved in 'data_slices/input'.


In [5]:
nn_input_all=None
nn_input=None